## Upload Vector Dataset to Vector Database

### Installing Utilities and Libraries

In [1]:
%pip install qdrant-client==1.14.3

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [2]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import json
import os
from dotenv import load_dotenv

load_dotenv()

QDRANT_URL = "https://731133a6-5b34-4bea-a857-363834c2fd43.us-west-1-0.aws.cloud.qdrant.io:6333"
COLLECTION_NAME = "carbonops_enterprise_knowledge"

qdrant_api_key = os.getenv("QDRANT_API_KEY")

### Upload Vectors to QDrant DB

In [3]:
qdrant_client = QdrantClient(
    url=QDRANT_URL, 
    api_key=qdrant_api_key,
)

if not qdrant_client.collection_exists(COLLECTION_NAME):
        qdrant_client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=VectorParams(size=1536, distance=Distance.DOT)
        )
        print(f"Collection '{COLLECTION_NAME}' created.")

        # Load vectors from file
        with open("vector-data.json", "r") as f:
            data = json.load(f)

        points = [
            PointStruct(
                id=i + 1,
                vector=entry["vector"],
                payload={"content": entry["chunk"]}
            ) for i, entry in enumerate(data)
        ]

        qdrant_client.upsert(
            collection_name=COLLECTION_NAME,
            wait=True,
            points=points
        )
        print(f"Upserted {len(points)} vectors.")
else:
        print(f"Collection '{COLLECTION_NAME}' already exists.")

/var/folders/qr/9_d4tppn2rlg_xxqp53j80w80000gp/T/ipykernel_64592/2938590045.py:1: UserWarning: Qdrant client version 1.14.3 is incompatible with server version 1.19.1. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  qdrant_client = QdrantClient(


Collection 'carbonops_enterprise_knowledge' created.
Upserted 25 vectors.
